> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Analytic and bootstrap uncertainty

VIGOR 2.12.1. A small bootstrap count checks execution, not SE calibration.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
store=vigor.open_store(ANALYSIS_STORE,mode="a",lazy=False)
K=vigor.open_grm(store,component="additive",materialize=True)
# Bound the example to six traits; increase BOOTSTRAPS for a precision study.
BOOTSTRAPS=int(os.environ.get('VIGOR_TUTORIAL_BOOTSTRAPS', os.environ.get('DOMINO_TUTORIAL_BOOTSTRAPS', 20)))
Y=train.iloc[:,:6]
common=dict(rank_G="full",rank_E="full",solver="lbfgs",progress=False,blas_threads=BLAS_THREADS,random_state=SEED)
a=vigor.genetic_varcov_PSD(K,Y,analytic_se="final",analytic_se_probes=16,analytic_trace_probes=8,n_boot=0,**common)
b=vigor.genetic_varcov_PSD(K,Y,analytic_se=False,n_boot=BOOTSTRAPS,bootstrap_n_jobs=2,bootstrap_blas_threads=1,**common)
print("bootstrap successes:",b.attrs.get("n_boot_success"))
print("SE relative difference:",np.linalg.norm(np.asarray(a["G_se"].to_pandas())-np.asarray(b["G_se"].to_pandas()))/max(np.linalg.norm(np.asarray(b["G_se"].to_pandas())),1e-12))
